# Microsoft Fabric shortcut inventory

This notebook scans Microsoft Fabric workspaces and builds a **complete inventory of OneLake shortcuts**, with quality and governance validations:

- **Discovers** workspaces and items (Lakehouses, KQL Databases, Mirrored Databases) via the [Fabric REST API](https://learn.microsoft.com/rest/api/fabric/).
- **Extracts** every shortcut from each item and resolves its target (internal OneLake or external storage: S3, ADLS Gen2, GCS, Dataverse, etc.).
- **Validates**: detects **orphaned** shortcuts (the target no longer exists within the scanned scope), **circular** ones (shortcut chains that form a cycle) and **ungoverned external** ones (no sensitivity label and no endorsement).
- **Presents** an interactive HTML report with filtered views and, optionally, **persists** the result to a Delta table for historical analysis.

**Requirements**: must run inside a Fabric notebook (it uses `notebookutils`, `sempy` and `displayHTML`). The user or service principal needs, at minimum, the *Viewer* role on the workspaces to be scanned. For `SCOPE_MODE = "all"` with full tenant coverage, and for governance enrichment, Fabric administrator permissions are required.

The flow is sequential: run the cells in order. All configuration is done in the parameters cell below.

## Parameters

The only cell you need to edit. Full reference:

| Parameter | Values | Description |
|---|---|---|
| `SCOPE_MODE` | `"all"` \| `"list"` | `"all"` scans every workspace in the tenant (via the admin API if you have permissions; otherwise the workspaces accessible to the user). `"list"` limits the scan to the workspaces in `WORKSPACE_LIST`. |
| `WORKSPACE_LIST` | list of `str` | Names or IDs (GUID) of the workspaces to scan. Only used with `SCOPE_MODE = "list"`. |
| `RESOLVE_NAMES` | `True` \| `False` | If `True`, the entries in `WORKSPACE_LIST` may be display names in addition to IDs; the notebook resolves the ID automatically. |
| `AUTH_MODE` | `"user"` \| `"sp"` | `"user"`: delegated token of the user running the notebook (on-behalf-of via `notebookutils`). `"sp"`: service principal, recommended for scheduled runs or full-tenant scans. |
| `SP_TENANT_ID` | GUID | Entra ID tenant ID. Only with `AUTH_MODE = "sp"`. |
| `SP_CLIENT_ID` | GUID | Client ID of the service principal's app registration. Only with `AUTH_MODE = "sp"`. |
| `SP_KEYVAULT` | URL | URL of the Azure Key Vault where the service principal secret is stored (the secret is never written into the notebook). |
| `SP_SECRET_NAME` | `str` | Name of the secret inside the Key Vault. |
| `GOVERNANCE_CHECK` | `True` \| `False` | If `True`, enriches each item with its sensitivity label and endorsement (via the `sempy` admin scan) and flags external shortcuts with no governance. Requires admin permissions; if missing, it degrades gracefully instead of failing. |
| `SAVE_TO_DELTA` | `True` \| `False` | If `True`, saves the final inventory to a Delta table in the attached Lakehouse (*overwrite* mode). |
| `DELTA_TABLE` | `str` | Name of the target Delta table. Only used with `SAVE_TO_DELTA = True`. |
| `MAX_WORKERS` | `int` | Number of parallel threads for the shortcuts API calls. Raise it for large scans; lower it if you hit frequent throttling (429). |

In [ ]:
# === Parameters ===
SCOPE_MODE       = "list"          # "all" | "list"
WORKSPACE_LIST   = ["Mi Workspace"]  # Only used if SCOPE_MODE = "list"
RESOLVE_NAMES    = True
AUTH_MODE        = "user"          # "user" | "sp"
SP_TENANT_ID     = ""
SP_CLIENT_ID     = ""
SP_KEYVAULT      = ""
SP_SECRET_NAME   = ""
GOVERNANCE_CHECK = True
SAVE_TO_DELTA    = False
DELTA_TABLE      = "shortcut_inventory"
MAX_WORKERS      = 8

In [ ]:
import datetime as _dt

def log(msg: str) -> None:
    print(f"[{_dt.datetime.now(_dt.timezone.utc).isoformat(timespec='seconds')}] {msg}")

log("Parameters loaded.")

## Authentication

Builds the HTTP client that the whole notebook will use, based on `AUTH_MODE`:

- **`_UserClient`** (`AUTH_MODE = "user"`): obtains a delegated token for the current user with `notebookutils.credentials.getToken()`. Requires no extra configuration, but only sees the workspaces the user has access to (`is_admin = False`).
- **`_SpClient`** (`AUTH_MODE = "sp"`): authenticates a service principal with `azure-identity`. The secret is read from Azure Key Vault at runtime — it is never stored in the notebook. It is assumed to have admin permissions (`is_admin = True`); if it does not, admin calls degrade without breaking the flow.

Both clients expose the same `get(url)` method, which refreshes the token on every call, so the rest of the notebook is agnostic to the authentication mode.

In [ ]:
import sempy.fabric as fabric

class _UserClient:
    """User token via notebookutils OBO — includes delegated Fabric API scopes."""
    def __init__(self):
        import requests as _req
        self._session = _req.Session()
        self.is_admin = False
    def _token(self):
        return notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    def get(self, url):
        if url.startswith("/"):
            url = "https://api.fabric.microsoft.com" + url
        self._session.headers["Authorization"] = f"Bearer {self._token()}"
        resp = self._session.get(url)
        resp.raise_for_status()
        return resp

class _SpClient:
    """Service principal client using azure-identity + requests."""
    def __init__(self, tenant_id, client_id, keyvault, secret_name):
        import requests
        from azure.identity import ClientSecretCredential
        secret = notebookutils.credentials.getSecret(keyvault, secret_name)
        self._cred = ClientSecretCredential(tenant_id, client_id, secret)
        self._session = requests.Session()
        self.is_admin = True  # assumed; admin calls still degrade gracefully if not
    def _token(self):
        return self._cred.get_token("https://api.fabric.microsoft.com/.default").token
    def get(self, url):
        if url.startswith("/"):
            url = "https://api.fabric.microsoft.com" + url
        self._session.headers["Authorization"] = f"Bearer {self._token()}"
        return self._session.get(url)

def build_client(auth_mode, sp_tenant_id, sp_client_id, sp_keyvault, sp_secret_name):
    if auth_mode == "sp":
        return _SpClient(sp_tenant_id, sp_client_id, sp_keyvault, sp_secret_name)
    return _UserClient()

client = build_client(AUTH_MODE, SP_TENANT_ID, SP_CLIENT_ID, SP_KEYVAULT, SP_SECRET_NAME)
log(f"Client built: auth_mode={AUTH_MODE}, is_admin={client.is_admin}")

## Workspace discovery

Resolves the list of workspaces to scan based on `SCOPE_MODE`:

- With `"all"`: it tries the admin API first (`/v1/admin/workspaces`, full tenant coverage) and, if the client is not an admin, falls back to `/v1/workspaces` (only the accessible workspaces).
- With `"list"`: it accepts IDs or, if `RESOLVE_NAMES = True`, display names, and validates them against the accessible workspaces. Entries that are not found are logged and skipped.

The `get_paged()` helper centralizes pagination (`continuationToken`) and retries on **HTTP 429** throttling with exponential backoff, honoring the `Retry-After` header. Every API call in the notebook goes through it.

In [ ]:
import time

def get_paged(client, url, value_key="value", max_retries=5):
    items, next_url = [], url
    while next_url:
        for attempt in range(max_retries):
            resp = client.get(next_url)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 2 ** attempt))
                log(f"429 on {next_url}; retrying in {wait}s")
                time.sleep(wait)
                continue
            break
        if resp.status_code < 200 or resp.status_code >= 300:
            log(f"GET {next_url} -> {resp.status_code}: {resp.text[:200]}")
            return items
        body = resp.json()
        items.extend(body.get(value_key, []))
        token = body.get("continuationToken")
        next_url = body.get("continuationUri") if token else None
    return items

In [ ]:
def resolve_workspaces(client, scope_mode, workspace_list, resolve_names):
    if scope_mode == "all":
        admin = get_paged(client, "/v1/admin/workspaces") if client.is_admin else []
        if admin:
            return [{"id": w["id"], "name": w.get("name") or w.get("displayName")} for w in admin]
        ws = get_paged(client, "/v1/workspaces")
        return [{"id": w["id"], "name": w.get("displayName")} for w in ws]
    # scope_mode == "list"
    all_ws = get_paged(client, "/v1/workspaces")
    by_id = {w["id"]: w.get("displayName") for w in all_ws}
    by_name = {w.get("displayName"): w["id"] for w in all_ws}
    out = []
    for entry in workspace_list:
        if entry in by_id:
            out.append({"id": entry, "name": by_id[entry]})
        elif resolve_names and entry in by_name:
            out.append({"id": by_name[entry], "name": entry})
        else:
            log(f"Workspace not found/accessible: {entry}")
    return out

workspaces = resolve_workspaces(client, SCOPE_MODE, WORKSPACE_LIST, RESOLVE_NAMES)
log(f"Resolved {len(workspaces)} workspace(s): {[w['name'] for w in workspaces]}")

## Item discovery

Lists all items in each workspace (`/v1/workspaces/{id}/items`) and flattens them into rows with workspace, ID, name and item type. This catalog is used later for two purposes: knowing which items to ask for their shortcuts, and validating whether OneLake targets exist within the scanned scope (orphan detection).

In [ ]:
def discover_items(client, workspaces):
    out = []
    for ws in workspaces:
        raw = get_paged(client, f"/v1/workspaces/{ws['id']}/items")
        for it in raw:
            out.append({
                "workspace_id": ws["id"],
                "workspace_name": ws["name"],
                "item_id": it["id"],
                "item_name": it.get("displayName"),
                "item_type": it.get("type"),
            })
    return out

items = discover_items(client, workspaces)
log(f"Discovered {len(items)} item(s) across {len(workspaces)} workspace(s)")

## Parsing shortcut targets

Normalizes the `target` object the API returns for each shortcut into a common flat schema. It distinguishes two cases:

- **OneLake** (internal): extracts the target's workspace, item and subpath, and builds a URI `onelake://workspace/item/path`. These are the only shortcuts that participate in orphan and cycle detection.
- **External** (Amazon S3, ADLS Gen2, Google Cloud Storage, S3 compatible, Dataverse, Azure Blob Storage, OneDrive/SharePoint): concatenates `location + subpath` and marks them with `is_external = True`, which makes them candidates for the governance review.

In [ ]:
_EXTERNAL_TYPES = {"AmazonS3", "AdlsGen2", "GoogleCloudStorage",
                   "S3Compatible", "Dataverse", "AzureBlobStorage",
                   "OneDriveSharePoint"}
# maps target_type -> key in the target dict holding its details
_DETAIL_KEY = {
    "OneLake": "oneLake", "AmazonS3": "amazonS3", "AdlsGen2": "adlsGen2",
    "GoogleCloudStorage": "googleCloudStorage", "S3Compatible": "s3Compatible",
    "Dataverse": "dataverse", "AzureBlobStorage": "azureBlobStorage",
    "OneDriveSharePoint": "oneDriveSharePoint",
}

def parse_shortcut_target(target):
    ttype = target.get("type")
    detail = target.get(_DETAIL_KEY.get(ttype, ""), {}) or {}
    if ttype == "OneLake":
        path = detail.get("path", "")
        wsid = detail.get("workspaceId")
        itid = detail.get("itemId")
        return {
            "target_type": "OneLake",
            "target_workspace_id": wsid,
            "target_item_id": itid,
            "target_subpath": path,
            "target_location": f"onelake://{wsid}/{itid}/{path}",
            "is_external": False,
        }
    location = detail.get("location", "")
    subpath = detail.get("subpath", "") or ""
    return {
        "target_type": ttype,
        "target_workspace_id": None,
        "target_item_id": None,
        "target_subpath": subpath,
        "target_location": f"{location}{subpath}",
        "is_external": ttype in _EXTERNAL_TYPES,
    }


## Shortcut extraction

Calls `/v1/workspaces/{ws}/items/{item}/shortcuts` for each item that supports shortcuts (only **Lakehouse**, **KQLDatabase** and **MirroredDatabase** expose this endpoint; the rest are skipped and logged). The calls are parallelized with a `ThreadPoolExecutor` of `MAX_WORKERS` threads to speed up large scans. If an item fails, a row with the `error` column is kept instead of aborting the entire scan.

In [ ]:
def _base_row(item):
    return {
        "workspace_id": item["workspace_id"],
        "workspace_name": item["workspace_name"],
        "item_id": item["item_id"],
        "item_name": item["item_name"],
        "item_type": item["item_type"],
        "shortcut_name": None, "shortcut_path": None,
        "target_type": None, "target_workspace_id": None,
        "target_item_id": None, "target_subpath": None,
        "target_location": None, "is_external": None,
    }

def extract_item_shortcuts(client, item):
    url = f"/v1/workspaces/{item['workspace_id']}/items/{item['item_id']}/shortcuts"
    try:
        scs = get_paged(client, url)
    except Exception as e:
        return [{**_base_row(item), "error": f"{type(e).__name__}: {e}"}]
    out = []
    for sc in scs:
        parsed = parse_shortcut_target(sc.get("target", {}))
        out.append({
            **_base_row(item),
            "shortcut_name": sc.get("name"),
            "shortcut_path": sc.get("path"),
            **parsed,
            "error": None,
        })
    return out

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import functools

# ponytail: only Fabric item types that expose the /shortcuts REST endpoint
_SHORTCUT_SUPPORTED_TYPES = {"Lakehouse", "KQLDatabase", "MirroredDatabase"}

def extract_all(client, items, max_workers):
    supported = [it for it in items if it["item_type"] in _SHORTCUT_SUPPORTED_TYPES]
    skipped = len(items) - len(supported)
    if skipped:
        log(f"Skipping {skipped} item(s) — type not in {_SHORTCUT_SUPPORTED_TYPES}")
    rows = []
    _extract = functools.partial(extract_item_shortcuts, client)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for res in ex.map(_extract, supported):
            rows.extend(res)
    return [r for r in rows if r["shortcut_name"] is not None or r["error"]]

rows = extract_all(client, items, MAX_WORKERS)
log(f"Extracted {len([r for r in rows if r['shortcut_name']])} shortcut(s); "
    f"{len([r for r in rows if r['error']])} item error(s)")


## Validation: orphaned shortcuts

Marks as **orphaned** (`is_orphan = True`) every OneLake shortcut whose target (workspace + item) does not exist among the discovered items, with the reason in `orphan_reason`. This usually indicates that the target item was deleted and the shortcut is now broken.

> **Note**: the check is done against the scanned scope. With `SCOPE_MODE = "list"`, a target living in a workspace outside the list will show up as orphaned even if it exists — widen the scope to confirm.

In [ ]:
def flag_orphans(rows, items):
    known = {(it["workspace_id"], it["item_id"]) for it in items}
    for r in rows:
        if r.get("target_type") == "OneLake":
            key = (r.get("target_workspace_id"), r.get("target_item_id"))
            if key in known:
                r["is_orphan"], r["orphan_reason"] = False, None
            else:
                r["is_orphan"] = True
                r["orphan_reason"] = (
                    f"OneLake target item {key[1]} in workspace {key[0]} "
                    "not found in scanned scope")
        else:
            r["is_orphan"], r["orphan_reason"] = False, None
    return rows


## Validation: circular shortcuts

Builds a directed graph where each OneLake shortcut is an edge *source item → target item* and searches for cycles using colored DFS (a back-edge to a node currently in progress indicates a cycle). Shortcuts whose edge participates in a cycle are marked with `is_circular = True` and the specific edge in `circular_path`. A cycle (A points to B and B points to A, directly or transitively) can cause infinite resolutions or duplicated data and should be broken.

In [ ]:
def detect_cycles(rows):
    # default
    for r in rows:
        r.setdefault("is_circular", False)
        r.setdefault("circular_path", None)
    # build adjacency from OneLake edges
    edges = {}  # src_node -> set(dst_node)
    def node(ws, it): return f"{ws}/{it}"
    for r in rows:
        if r.get("target_type") == "OneLake" and r.get("target_item_id"):
            s = node(r["workspace_id"], r["item_id"])
            d = node(r["target_workspace_id"], r["target_item_id"])
            edges.setdefault(s, set()).add(d)
    # find nodes that participate in any cycle (Tarjan-lite via DFS)
    WHITE, GRAY, BLACK = 0, 1, 2
    color, cyclic_edges = {}, set()
    def dfs(u, path):
        color[u] = GRAY
        path.append(u)
        for v in edges.get(u, ()):  # noqa
            if color.get(v, WHITE) == GRAY:        # back-edge -> cycle
                i = path.index(v)
                for a, b in zip(path[i:], path[i+1:] + [v]):
                    cyclic_edges.add((a, b))
            elif color.get(v, WHITE) == WHITE:
                dfs(v, path)
        path.pop()
        color[u] = BLACK
    for n in list(edges):
        if color.get(n, WHITE) == WHITE:
            dfs(n, [])
    # mark rows whose edge is part of a cycle
    for r in rows:
        if r.get("target_type") == "OneLake" and r.get("target_item_id"):
            s = node(r["workspace_id"], r["item_id"])
            d = node(r["target_workspace_id"], r["target_item_id"])
            if (s, d) in cyclic_edges:
                r["is_circular"] = True
                r["circular_path"] = f"{s} -> {d}"
    return rows


## Governance enrichment

Only runs if `GOVERNANCE_CHECK = True`. It fetches item governance metadata via the `sempy` *admin scan* (`fabric.admin.scan_workspaces`, requires Fabric administrator permissions) and adds to each row:

- `item_sensitivity_label`: sensitivity label of the item that contains the shortcut.
- `item_endorsement`: endorsement of the item (`Certified` or `Promoted`).
- `governance_flag = True` when an **external** shortcut lives in an item with no sensitivity label and no endorsement — that is, data leaving or entering external sources with no declared governance control whatsoever.

If the scan is not available (missing permissions or API), it is logged and the notebook continues without enrichment.

In [ ]:
_GOOD_ENDORSEMENTS = {"Certified", "Promoted"}

def apply_governance(rows, gov):
    for r in rows:
        meta = gov.get(r.get("item_id"), {})
        sens = meta.get("sensitivity")
        endo = meta.get("endorsement")
        r["item_sensitivity_label"] = sens
        r["item_endorsement"] = endo
        if r.get("is_external"):
            r["governance_flag"] = not sens and endo not in _GOOD_ENDORSEMENTS
        else:
            r["governance_flag"] = False
    return rows


In [ ]:
def fetch_governance(workspaces):
    gov = {}
    try:
        import sempy.fabric as fabric
        ids = [w["id"] for w in workspaces][:100]
        df = fabric.admin.scan_workspaces(workspace=ids, return_dataframe=True)
    except Exception as e:
        log(f"Governance scan unavailable: {type(e).__name__}: {e}")
        return gov
    # df has one row per workspace; each contains item collections. Flatten lakehouse-like items.
    for _, ws_row in df.iterrows():
        for coll in ("Lakehouses", "Datasets", "Items"):
            for it in (ws_row.get(coll) or []):
                iid = it.get("id") or it.get("objectId")
                if iid:
                    gov[iid] = {
                        "sensitivity": (it.get("sensitivityLabel") or {}).get("labelId")
                                        if isinstance(it.get("sensitivityLabel"), dict)
                                        else it.get("sensitivityLabel"),
                        "endorsement": (it.get("endorsementDetails") or {}).get("endorsement")
                                        if isinstance(it.get("endorsementDetails"), dict)
                                        else it.get("endorsement"),
                    }
    return gov

gov = fetch_governance(workspaces) if GOVERNANCE_CHECK else {}
rows = apply_governance(rows, gov)
log(f"Governance: {len(gov)} item(s) enriched; "
    f"{len([r for r in rows if r.get('governance_flag')])} flagged")


## Name resolution and finalization

Converts the GUIDs of OneLake targets into readable names (`target_workspace_name`, `target_item_name`) and builds `target_location_display`, a readable URI like `onelake://Sales/SalesLakehouse/Tables/customers`. It first looks in the already-discovered catalog; if the target is outside the scanned scope, it does a one-off API lookup (cached to avoid repeated calls).

Then, `finalize()` stamps each row with `scan_timestamp` (UTC) and guarantees that all rows share the same column schema (`_ALL_COLS`), filling missing values with `None`.

In [ ]:
_ALL_COLS = ["scan_timestamp","workspace_id","workspace_name","item_id","item_name",
             "item_type","shortcut_name","shortcut_path","target_type","target_workspace_id",
             "target_workspace_name","target_item_id","target_item_name","target_subpath",
             "target_location","target_location_display","is_external","is_orphan","orphan_reason",
             "is_circular","circular_path","item_sensitivity_label","item_endorsement",
             "governance_flag","error"]

def resolve_target_names(rows, items, workspaces, client=None):
    """Resolve OneLake target workspace/item ids to display names.

    Falls back to a live lookup via `client` when the target is outside the
    scanned scope (e.g. a shortcut pointing to a workspace not in
    WORKSPACE_LIST), so cross-workspace targets still get readable names.
    """
    item_name = {(it["workspace_id"], it["item_id"]): it["item_name"] for it in items}
    ws_name = {w["id"]: w["name"] for w in workspaces}
    _ws_cache, _item_cache = {}, {}

    def _fetch_ws_name(wsid):
        if not wsid or client is None:
            return None
        if wsid not in _ws_cache:
            try:
                _ws_cache[wsid] = client.get(f"/v1/workspaces/{wsid}").json().get("displayName")
            except Exception as e:
                log(f"Could not resolve workspace name for {wsid}: {type(e).__name__}: {e}")
                _ws_cache[wsid] = None
        return _ws_cache[wsid]

    def _fetch_item_name(wsid, itid):
        if not wsid or not itid or client is None:
            return None
        key = (wsid, itid)
        if key not in _item_cache:
            try:
                _item_cache[key] = client.get(
                    f"/v1/workspaces/{wsid}/items/{itid}").json().get("displayName")
            except Exception as e:
                log(f"Could not resolve item name for {itid} in {wsid}: {type(e).__name__}: {e}")
                _item_cache[key] = None
        return _item_cache[key]

    for r in rows:
        if r.get("target_type") == "OneLake":
            wsid, itid = r.get("target_workspace_id"), r.get("target_item_id")
            r["target_workspace_name"] = ws_name.get(wsid) or _fetch_ws_name(wsid)
            r["target_item_name"] = item_name.get((wsid, itid)) or _fetch_item_name(wsid, itid)
            wsn = r["target_workspace_name"] or wsid or "?"
            itn = r["target_item_name"] or itid or "?"
            subpath = r.get("target_subpath") or ""
            r["target_location_display"] = f"onelake://{wsn}/{itn}/{subpath}"
        else:
            r["target_workspace_name"] = None
            r["target_item_name"] = None
            r["target_location_display"] = r.get("target_location")
    return rows

def finalize(rows):
    ts = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")
    for r in rows:
        r["scan_timestamp"] = ts
        for c in _ALL_COLS:
            r.setdefault(c, None)
    return rows

final_rows = finalize(resolve_target_names(rows, items, workspaces, client))
log(f"Finalized {len(final_rows)} row(s)")

## HTML report

Renders the inventory as an interactive report inside the notebook itself (`displayHTML`), with no external dependencies. It includes:

- **Summary**: totals of shortcuts, orphaned, circular, governance flags and a breakdown by target type.
- **Four tabbed views**: *Overview* (everything), *Circular*, *Orphan* and *Governance*, each with its own counter.
- **Per-row color coding**: red = orphaned, orange = circular, yellow = governance flag.

All content is escaped with `html.escape()` before rendering.

In [ ]:
import html as _html_mod

def _row_color(r):
    if r.get("is_orphan"):       return "#f8d7da"
    if r.get("is_circular"):     return "#ffe5b4"
    if r.get("governance_flag"): return "#fff3cd"
    return "#ffffff"

_COLS = ["workspace_name","item_name","shortcut_name","shortcut_path",
         "target_type","target_location_display","target_workspace_name","target_item_name",
         "is_orphan","is_circular","governance_flag","item_endorsement","error"]

def _build_table(rows):
    if not rows:
        return "<p style='color:#666;font-style:italic'>No items in this view.</p>"
    head = "".join(
        f"<th style='text-align:left;padding:6px 8px;border-bottom:2px solid #dee2e6;"
        f"background:#f8f9fa;position:sticky;top:0'>{c}</th>" for c in _COLS)
    body = []
    for r in rows:
        cells = "".join(
            f"<td style='padding:5px 8px;border-bottom:1px solid #e9ecef'>"
            f"{_html_mod.escape(str(r.get(c) or ''))}</td>" for c in _COLS)
        body.append(f"<tr style='background:{_row_color(r)}'>{cells}</tr>")
    return (
        "<div style='overflow-x:auto;max-height:500px;overflow-y:auto'>"
        "<table style='border-collapse:collapse;font-family:sans-serif;"
        "font-size:12px;width:100%'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )

def render_html(rows):
    total      = len(rows)
    n_orphan   = sum(1 for r in rows if r.get("is_orphan"))
    n_circular = sum(1 for r in rows if r.get("is_circular"))
    n_gov      = sum(1 for r in rows if r.get("governance_flag"))
    by_type    = {}
    for r in rows:
        t = r.get("target_type") or "unknown"
        by_type[t] = by_type.get(t, 0) + 1

    views = [
        ("overview",   "Overview",   rows,                                          "#0078d4"),
        ("circular",   "Circular",   [r for r in rows if r.get("is_circular")],     "#d97706"),
        ("orphan",     "Orphan",     [r for r in rows if r.get("is_orphan")],       "#dc2626"),
        ("governance", "Governance", [r for r in rows if r.get("governance_flag")], "#7c3aed"),
    ]

    btn_css = (
        "display:inline-flex;align-items:center;gap:6px;padding:8px 16px;"
        "border:none;border-radius:6px;cursor:pointer;font-size:13px;"
        "font-family:sans-serif;font-weight:500;transition:opacity .15s"
    )

    buttons, panels = [], []
    for i, (vid, label, vrows, color) in enumerate(views):
        active = f"background:{color};color:#fff;box-shadow:0 2px 6px {color}66"
        idle   = "background:#f3f4f6;color:#374151"
        cnt    = len(vrows)
        buttons.append(
            f"<button id='btn-{vid}' onclick=\"showView('{vid}')\" "
            f"style='{btn_css};{active if i == 0 else idle}'>"
            f"{label}"
            f"<span style='background:rgba(255,255,255,0.25);border-radius:12px;"
            f"padding:2px 7px;font-size:11px'>{cnt}</span>"
            f"</button>"
        )
        display = 'block' if i == 0 else 'none'
        panels.append(
            f"<div id='panel-{vid}' style='display:{display}'>"
            f"{_build_table(vrows)}</div>"
        )

    summary = (
        "<div style='font-family:sans-serif;font-size:13px;color:#374151;"
        "margin-bottom:14px;padding:10px 14px;background:#f8f9fa;border-radius:6px;"
        "border-left:4px solid #0078d4'>"
        f"<b>Total shortcuts:</b> {total} &nbsp;·&nbsp; "
        f"<b>Orphan:</b> {n_orphan} &nbsp;·&nbsp; "
        f"<b>Circular:</b> {n_circular} &nbsp;·&nbsp; "
        f"<b>Governance flags:</b> {n_gov}<br>"
        "<span style='color:#6b7280'><b>By target type:</b> "
        + ", ".join(f"{k}: {v}" for k, v in sorted(by_type.items()))
        + "</span></div>"
    )

    colors_js = (
        "{"
        + ",".join(f"'{vid}':'{color}'" for vid, _, __, color in views)
        + "}"
    )
    vids_js = str([vid for vid, *_ in views])

    script = (
        "<script>\n"
        f"var _c={colors_js};var _v={vids_js};\n"
        "function showView(id){\n"
        "  _v.forEach(function(v){\n"
        "    document.getElementById('panel-'+v).style.display=v===id?'block':'none';\n"
        "    var b=document.getElementById('btn-'+v);\n"
        "    if(v===id){b.style.background=_c[v];b.style.color='#fff';"
        "b.style.boxShadow='0 2px 6px '+_c[v]+'66';}"
        "else{b.style.background='#f3f4f6';b.style.color='#374151';"
        "b.style.boxShadow='none';}});\n"
        "}\n"
        "</script>"
    )

    btn_bar = (
        "<div style='display:flex;gap:8px;margin-bottom:14px;flex-wrap:wrap'>"
        + "".join(buttons)
        + "</div>"
    )
    legend = (
        "<div style='font-family:sans-serif;font-size:11px;margin-top:10px;display:flex;gap:12px'>"
        "<span style='background:#f8d7da;padding:2px 8px;border-radius:4px'>orphan</span>"
        "<span style='background:#ffe5b4;padding:2px 8px;border-radius:4px'>circular</span>"
        "<span style='background:#fff3cd;padding:2px 8px;border-radius:4px'>governance</span>"
        "</div>"
    )

    return (
        "<div style='padding:16px'>"
        + summary + btn_bar + "".join(panels) + legend + script
        + "</div>"
    )

In [ ]:
displayHTML(render_html(final_rows))

## Optional: save to a Delta table

Only runs if `SAVE_TO_DELTA = True`. It converts the inventory into a Spark DataFrame with the fixed `_ALL_COLS` schema and writes it to the `DELTA_TABLE` table in the attached Lakehouse in **overwrite** mode (each run replaces the previous one; `scan_timestamp` identifies the scan). Useful for querying the inventory with SQL, building Power BI reports on top, or — by switching the mode to `append` — keeping a history of scans.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType

_BOOL_COLS = {"is_external", "is_orphan", "is_circular", "governance_flag"}

_SPARK_SCHEMA = StructType([
    StructField(c, BooleanType() if c in _BOOL_COLS else StringType(), True)
    for c in _ALL_COLS
])

def to_spark_df(rows):
    # explicit schema avoids CANNOT_DETERMINE_TYPE when a column is all-None
    # across every row (type inference has nothing to sample from)
    ordered = [{c: r.get(c) for c in _ALL_COLS} for r in rows]
    return spark.createDataFrame(ordered, schema=_SPARK_SCHEMA) if ordered else None

if SAVE_TO_DELTA:
    sdf = to_spark_df(final_rows)
    if sdf is not None:
        sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(DELTA_TABLE)
        log(f"Saved {sdf.count()} row(s) to Delta table '{DELTA_TABLE}' (overwrite)")
    else:
        log("No rows to save; skipped Delta write.")
else:
    log("SAVE_TO_DELTA is False; skipped Delta write.")
